# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HasanKhan05/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am provisionally choosing the predefined **Lane 2: Refresh / Content Opportunity Scoring**. My search question is: **Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?** This lane fits the starter data because it contains page-level search exposure, engagement, lifecycle, and observed movement signals. The goal is not merely to train a model: I will first build an explainable rule baseline, inspect the evidence, and use a model only if it improves a held-out ranked review queue while remaining understandable.

In [1]:
from pathlib import Path
import pandas as pd

data_file = next(
    candidate / "data/raw/content_refresh_anonymized.csv"
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data/raw/content_refresh_anonymized.csv").exists()
)
starter = pd.read_csv(data_file)
starter["is_declining_proxy"] = starter["trend_direction"].str.lower().eq("down")
print("Loaded the public-safe starter snapshot for the Refresh / Content Opportunity lane.")


Loaded the public-safe starter snapshot for the Refresh / Content Opportunity lane.


## 2. The question: decision, action, cost of a wrong call

**Decision and unit of analysis.** One row represents one pseudonymized content item at a trailing-90-day snapshot. For a content editor deciding what to inspect next, I will produce a ranked review queue with a priority score, suggested action, reason codes, and confidence. The editor—not the score—then decides whether to refresh, expand, protect, prune/consolidate, or monitor the page.

**Cost of a wrong call.** A false positive wastes limited editorial time and may encourage an unnecessary change to a healthy page. A false negative leaves a measurable decline or opportunity unreviewed. Because both errors matter, the queue should be compared with a transparent rule baseline using a top-K metric that matches the editor's review capacity. The output is decision-support, not automatic publishing or proof that an edit will improve performance.

In [2]:
review_capacity = 50
reviewable = starter.loc[starter["impressions_90d"] >= 100].copy()
print(f"Provisional operating policy: rank {len(reviewable):,} measurable pages, then review the top {review_capacity}.")
print("This threshold is a starting policy choice and will be tested, not treated as universal.")


Provisional operating policy: rank 22,006 measurable pages, then review the top 50.
This threshold is a starting policy choice and will be tested, not treated as universal.


## 3. Quick look at the data (2-3 real numbers)

The starter snapshot is large enough to show a prioritization problem rather than a hand-picked example. It contains **30,000 content items across 32 pseudonymized clients**. After a provisional minimum-volume filter of 100 impressions, **22,006 items remain**; **13,152 of them (59.8%)** have the observed current-decline proxy. That is far more than an editor can inspect manually, so the next seven weeks can test whether a transparent score—and then, only if justified, ML—can produce a more useful top-of-queue than a fixed rule.

In [3]:
declining_reviewable = int(reviewable["is_declining_proxy"].sum())
declining_share = reviewable["is_declining_proxy"].mean()

print(f"1) Starter snapshot: {len(starter):,} content items across {starter['client_id'].nunique()} pseudonymized clients")
print(f"2) Items with at least 100 impressions: {len(reviewable):,}")
print(f"3) Observed declining items in that slice: {declining_reviewable:,} ({declining_share:.1%})")


1) Starter snapshot: 30,000 content items across 32 pseudonymized clients
2) Items with at least 100 impressions: 22,006
3) Observed declining items in that slice: 13,152 (59.8%)


## 4. Careful words: what I can and can't claim

This snapshot can support **observed, measured, directional, and decision-support** statements: for example, that certain page characteristics are associated with the current decline proxy, or that one ranking performs better than a baseline on held-out data. It cannot show that a refresh caused recovery, identify Google's ranking factors, or guarantee that a recommended action will work. The starter `trend_direction` field describes current-window movement and is only a provisional proxy; it and `trend_pct` will never be features. A stronger capstone test will use earlier feature windows and a later observed outcome, with client-grouped and time-aware validation. Pseudonymized IDs are for grouping and splitting only, never model inputs or attempts to identify clients.

In [4]:
assert starter["content_id"].is_unique, "Expected one row per pseudonymized content item"
assert starter["client_id"].nunique() == 32, "Unexpected starter-client count"
assert not reviewable.empty, "The provisional lane slice should not be empty"
print("Checks passed: page grain is unique, identifiers remain pseudonymous, and the lane slice is non-empty.")
print("Interpretation remains observational and decision-support only.")


Checks passed: page grain is unique, identifiers remain pseudonymous, and the lane slice is non-empty.
Interpretation remains observational and decision-support only.


## 5. Self-check

- [x] I chose one predefined lane and explained why it is worth investigating.
- [x] I named the decision owner, action, output, unit of analysis, and costs of wrong calls.
- [x] The starter CSV is loaded and at least two real supporting numbers are shown.
- [x] I explained why the work begins with a baseline and evidence rather than just training a model.
- [x] Claims use careful observed, measured, directional, and decision-support language.
- [x] No client names, domains, URLs, private queries, or identifying examples are displayed.
- [x] The notebook has been run top to bottom with no errors before submission.